# 06 · Conclusiones

## Pregunta central

> **¿Qué patrones temporales explican cómo una canción entra, crece y desaparece del
> chart en Argentina, y qué tan predecible es ese comportamiento?**

Integramos acá lo que salió de las tres sub-preguntas (notebooks 03, 04 y 05).

## Respuesta integrada

**1. La entrada, el crecimiento y la desaparición siguen un patrón temporal nítido.**
Tratada como señal, la trayectoria de una canción en el Top 200 tiene dos formas de onda
arquetípicas (notebook 03):

- **Hit viral** — un **transitorio no estacionario**: entrada abrupta, **pico en ~3 días**
  (pendiente de subida ~63× la de caída) y **caída sostenida** durante meses. Su energía
  espectral se concentra en **baja frecuencia** (domina el evento de lanzamiento) y su
  autocorrelación decae lento.
- **Tema de catálogo** — una **meseta cuasi-estacionaria**: entra sin salto, se sostiene
  años en una banda angosta de streams y deja ver mejor la **estacionalidad semanal**
  (más consumo los fines de semana).

Sobre ambas, el **filtrado MA7/MA21** separa limpiamente **tendencia** (ciclo de vida) de
**componente semanal** y **ruido**, y la energía se conserva (Parseval) al filtrar. La
serie viral es, formalmente, **no estacionaria** (ADF/KPSS), lo que la conecta con el
**procesamiento tiempo-frecuencia**: su contenido espectral cambia con el tiempo (pico vs.
cola), por eso el análisis de Fourier global se complementa con filtrado local.

**2. El comportamiento argentino está fuertemente alineado con el global, casi sin
desfasaje.** (notebook 04) En un día típico hay **~65 artistas en común** entre el Top 200
de Argentina y el Global, con un pico histórico en **2017** (crossover latino). La
**correlación cruzada** de la actividad AR vs. Global es débil y **casi simétrica alrededor
de lag 0**: Argentina **no lidera ni va a la zaga, acompaña al mundo casi en tiempo real**,
conservando un margen local (cumbia, rock nacional, trap) que le da identidad parcial.

**3. Qué tan predecible es: poco, más allá del baseline naive.** (notebook 05) Acá está el
hallazgo que **no hay que suavizar**. Se modeló la serie viral (no estacionaria → `d=1`;
orden elegido a mano por ACF/PACF y por AIC → **ARIMA(1,1,1)**) y se la comparó con
**Prophet** y con un **baseline naive** (repetir el último valor) en **dos backtests**:

| Tramo de test | Naive | ARIMA(1,1,1) | Prophet |
|---|---|---|---|
| **A · meseta plana** (últimos 25 días) | 2.358 | **2.358** | 106.595 |
| **B · caída post-pico** (días 20-44) | 16.748 | **16.760** | 511.797 |

**En ningún tramo los modelos superaron al naive.** En la meseta el empate es trivial
(serie ya plana). En la caída —donde **sí hay dinámica real**— ARIMA(1,1,1) **vuelve a
empatar** al naive: al ser un random-walk **sin término de deriva**, pronostica un valor
constante y arrastra la bajada igual que la persistencia. **Prophet, además, empeora
bastante** en ambos tramos (sobre-extrapola con pocos datos). Es decir: **esta serie viral
no fue bien capturada por ARIMA(1,1,1) ni por Prophet**; para superar al naive haría falta
un modelo con **término de deriva explícito** (ARIMA con tendencia, o modelar el
decaimiento exponencial / log-streams).

**Síntesis.** El ciclo de vida de una canción en Argentina es **estructuralmente muy
legible** (entrada → pico → caída, con estacionalidad semanal y fuerte acople al global),
pero a nivel de **valor diario puntual es poco predecible**: las herramientas estándar de
forecasting no le ganan a la simple persistencia. La regularidad está en la **forma**, no
en el detalle día a día.

## Temas del programa considerados pero no usados

> ⚠️ **Para el equipo:** esta tabla se armó con los temas y razones que ya figuran en los
> notebooks (nota de wavelets en el 03, nota de redes en el 05) porque **no se encontró el
> archivo `cobertura_temario.md` en la raíz del proyecto**. Reemplazar / completar con el
> contenido definitivo de ese archivo cuando esté disponible.

| Tema del programa | Por qué se descartó |
|---|---|
| **Muestreo en amplitud (cuantización)** | Los datos ya vienen discretizados en el tiempo (1 muestra/día) y con valores enteros de streams; no hay una etapa de conversión analógico-digital ni de cuantización de amplitud que analizar en esta fuente de datos. |
| **Filtros IIR** | Para separar tendencia / estacionalidad / ruido alcanzó con filtros de **media móvil (FIR)**: son de fase lineal, fáciles de interpretar y sin problemas de estabilidad. Un IIR habría agregado complejidad (diseño de polos, fase no lineal) sin beneficio para este alcance. |
| **Wavelets (algoritmo de Mallat)** | Se consideró para separar el pico del resto sin perder localización temporal, pero el **filtrado MA7/MA21 ya resuelve** esa separación tiempo-frecuencia para el alcance del trabajo. |
| **Redes neuronales (RNN / LSTM)** | Con ~200 puntos diarios **no hay volumen suficiente** para entrenar una red sin sobreajustar; modelos clásicos (ARIMA/Prophet) aprovechan mejor una muestra chica. |
| **Redes convolucionales (CNN)** | No aplican: no hay **estructura espacial** (ni de vecindad tipo imagen) en una serie temporal univariada como esta. |

## Cierre

Como señal, una canción en el chart argentino es un **evento con forma reconocible** que el
instrumental de la materia describe bien (Fourier, filtrado, energía, correlación); su
**predecibilidad puntual, en cambio, es limitada**, y ese límite —ningún modelo probado
supera al naive— es en sí mismo uno de los resultados del trabajo.